# D8 · SQL para *features* — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea B · *Data Scientist* · Semana 9 · Se ejecuta en Google Colab.\n\n> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que te dice ✅ si está bien o te da una pista si todavía no.

---

## ¿Qué vas a lograr?

En el tronco común aprendiste a **consultar** datos con SQL (M5): pedirle a una base
que te muestre lo que ya está guardado. En este módulo das un paso clave para la
ciencia de datos: usar SQL no para *mirar* los datos, sino para **fabricar variables
nuevas** que después alimentarán un modelo. A esas variables se les llama
***features*** (en español, *características* o *atributos*).

**Competencia de salida:** a partir de una tabla de eventos en bruto, construir con
SQL una **tabla analítica** —una fila por entidad, varias columnas de *features*—
lista para entrenar un modelo, usando agregaciones, lógica condicional y una variable
temporal de rezago, y entendiendo qué es la *fuga de información* (*data leakage*).

**Dato real con el que trabajaremos:** los **15 últimos sismos** publicados por el
**Centro Sismológico Nacional (CSN)** el 2026-06-18. Datos públicos, de
[sismologia.cl](https://www.sismologia.cl/).

**Entregable:** que las **4 celdas de chequeo** de este cuaderno muestren ✅.


## Antes de empezar: cambiamos de tema (a propósito)

Hasta aquí el hilo conductor del bootcamp fueron las **compras públicas**. En la Línea B
abrimos la cancha a otros dominios del Estado, porque un *data scientist* tiene que
sentirse cómodo con datos de cualquier tipo. Hoy el dato es **sísmico**, y no es un
capricho: Chile es uno de los países más sísmicos del mundo, los datos del CSN son
abiertos, y una tabla de sismos es el ejemplo *perfecto* para aprender a fabricar
*features*. Cada sismo es un evento con magnitud, profundidad, lugar y hora: materia
prima ideal para inventar variables.


## 1. La idea central: de *eventos* a *features*

### ¿Qué es una *feature*?
Una *feature* es **una columna que describe algo y que un modelo puede usar para
aprender**. Si más adelante quisieras un modelo que estime "¿qué tan activa
sísmicamente está una región?", el modelo no puede "mirar" la realidad: solo ve
**números en columnas**. Tu trabajo es traducir el fenómeno a columnas útiles:
*cuántos* sismos tuvo la región, su magnitud *promedio*, la *máxima*, cuántos fueron
*superficiales*… Cada una de esas columnas es una *feature*.

> 🧠 **Analogía.** Imagina que describes a una persona para que alguien que no la ve
> pueda reconocerla: estatura, edad, color de pelo. Esas descripciones son *features*.
> El modelo es ese "alguien que no ve": solo conoce el mundo a través de las *features*
> que tú le pasas. Buenas *features* → buen modelo. *Features* pobres → ni el mejor
> algoritmo lo arregla. Por eso se dice que **fabricar *features* es el 80% del trabajo**.

### Tres palabras que usaremos todo el módulo
- **Evento / observación en bruto:** una fila de los datos crudos. Aquí, **un sismo**.
- **Entidad:** la *cosa* sobre la que queremos decidir o predecir, y que tendrá **una
  fila** en la tabla final. Aquí elegimos la **región**.
- **Tabla analítica** (o *Analytical Base Table*, **ABT**): la tabla final con
  **una fila por entidad** y **una columna por *feature***. Es lo que se le entrega a
  un modelo. Construirla es la meta de hoy.

### El viaje de hoy
```
tabla 'sismos'            agregar / clasificar / rezagar       TABLA ANALITICA
(1 fila = 1 sismo)   ───────────────  con SQL  ──────────▶   (1 fila = 1 region,
                                                              varias features)
```
SQL es buenísimo para este viaje porque `GROUP BY`, `CASE WHEN` y las funciones de
ventana resumen miles de filas en las pocas columnas que el modelo necesita.

Ejecuta la siguiente celda para cargar los datos reales y ver la tabla `sismos`.


In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os
import urllib.request
import sqlite3
import pandas as pd

# Si el archivo no existe localmente (ej: en Colab), intentamos descargarlo desde GitHub
if not os.path.exists("sismos.csv"):
    try:
        url = "https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/B1-sql-para-features/sismos.csv"  # Reemplazar por la URL real del repositorio al publicar
        urllib.request.urlretrieve(url, "sismos.csv")
        print("sismos.csv descargado automáticamente.")
    except Exception:
        print("Aviso: No se pudo descargar sismos.csv automáticamente. Si estás en Colab, asegúrate de subirlo manualmente.")

# Cargamos el archivo CSV usando pandas
df_sismos = pd.read_csv("sismos.csv")

# Creamos una base de datos SQLite en memoria
con = sqlite3.connect(":memory:")

# Volcamos el DataFrame en la tabla "sismos" de la base de datos
df_sismos.to_sql("sismos", con, if_exists="replace", index=False)

# Atajo para correr consultas y mostrarlas como tablas formateadas
def correr(sql):
    return pd.read_sql_query(sql, con)

print("Listo. Tabla \x27sismos\x27 creada en la base de datos con", len(df_sismos), "eventos reales.")
correr("SELECT * FROM sismos")

## 2. *Features* de agregación con `GROUP BY`

La forma más común de fabricar *features* es **resumir muchas filas en una sola por
entidad**. Eso es exactamente lo que hace `GROUP BY`: junta las filas que comparten un
valor (por ejemplo, todas las de la región `Coquimbo`) y aplica una **función de
agregación** a cada grupo.

Las funciones de agregación más útiles:

| Función | Qué calcula | *Feature* de ejemplo |
| --- | --- | --- |
| `COUNT(*)` | cuántas filas hay en el grupo | número de sismos de la región |
| `SUM(col)` | suma | total de algo |
| `AVG(col)` | promedio | magnitud promedio |
| `MAX(col)` / `MIN(col)` | mayor / menor | magnitud máxima registrada |

Forma general:
```sql
SELECT  columna_que_agrupa,
        COUNT(*)   AS n,
        AVG(otra)  AS promedio
FROM    tabla
GROUP BY columna_que_agrupa
```

> ⚠️ **Errores típicos**
> - **Pedir una columna que no está en el `GROUP BY` ni dentro de una función de
>   agregación.** Si agrupas por `region`, no puedes pedir `lugar` "suelto": cada región
>   tiene muchos lugares y SQL no sabría cuál mostrar. Regla: lo que va en el `SELECT` o
>   está en el `GROUP BY`, o está dentro de `COUNT/SUM/AVG/...`.
> - **Confundir `COUNT(*)` con `SUM(...)`.** `COUNT(*)` cuenta *filas*; `SUM(magnitud)`
>   *suma* las magnitudes (un número enorme sin sentido aquí).
> - **Olvidar nombrar la columna** con `AS`. Sin alias te queda un encabezado feo como
>   `AVG(magnitud)` y, peor, las celdas de chequeo no la encontrarán.


### ✍️ Ejercicio 1 — Resumen sísmico por región
Crea una consulta que devuelva **una fila por región** con estas cinco columnas
(usa exactamente estos nombres con `AS`):

`region`, `n_sismos`, `magnitud_promedio`, `magnitud_maxima`, `profundidad_promedio`.

Ordénala por `n_sismos` de mayor a menor. Completa `consulta_1` y ejecuta.


In [ ]:
consulta_1 = """
-- TODO: reemplaza el SELECT de abajo por tu consulta.
-- Objetivo: una fila por region con estas 5 columnas (nombres EXACTOS, con AS):
--   region, n_sismos, magnitud_promedio, magnitud_maxima, profundidad_promedio
-- Pistas: GROUP BY region; COUNT(*) cuenta filas; AVG() promedia; MAX() toma el mayor;
--         ROUND(AVG(...), 2) redondea. Ordena por n_sismos DESC.
SELECT * FROM sismos
"""
correr(consulta_1)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ───────────────────────────────────────────
try:
    df = correr(consulta_1)
    cols = set(df.columns)
    esperadas = {"region", "n_sismos", "magnitud_promedio",
                 "magnitud_maxima", "profundidad_promedio"}
    assert esperadas.issubset(cols), \
        f"Faltan columnas o tienen otro nombre. Necesito: {sorted(esperadas)}"
    assert len(df) == 6, f"Deberian salir 6 regiones, no {len(df)}. Te falto GROUP BY?"
    assert int(df["n_sismos"].sum()) == 15, "Los n_sismos deben sumar 15 (total de eventos)."
    fila_anto = df[df["region"] == "Antofagasta"].iloc[0]
    assert int(fila_anto["n_sismos"]) == 6, "Antofagasta debe tener 6 sismos."
    assert abs(float(fila_anto["magnitud_maxima"]) - 3.6) < 1e-6, \
        "La magnitud maxima de Antofagasta es 3.6 (el sismo de Ollague)."
    print("✅ Correcto: construiste tus primeras features de agregacion por region.")
except Exception as e:
    print("❌ Aun no. Revisa tu consulta_1 y vuelve a intentar.")
    print("   Pista:", e)

## 3. *Features* condicionales con `CASE WHEN`

A veces la *feature* útil no es un promedio, sino una **etiqueta** o una **bandera**
que resume una regla. Para eso sirve `CASE WHEN`: es el "si… entonces…" de SQL.

```sql
CASE
    WHEN profundidad_km < 70   THEN 'superficial'
    WHEN profundidad_km <= 300 THEN 'intermedio'
    ELSE 'profundo'
END AS profundidad_tipo
```

SQL evalúa las condiciones **de arriba hacia abajo** y se queda con la **primera** que
se cumple. Por eso el orden importa: un sismo de 50 km entra por la primera línea y ya
no se evalúan las demás.

¿Por qué es tan útil para un modelo? Porque muchos modelos trabajan mejor con
**categorías limpias** o con **banderas 0/1** ("¿fue perceptible?", "¿es de gran
ciudad?") que con un número crudo. `CASE WHEN` es la herramienta para crearlas.

> 🧠 **Analogía.** Es como los rangos de notas: en vez del puntaje exacto, a veces te
> sirve más la etiqueta "aprobado / reprobado". `CASE WHEN` convierte un número continuo
> en la etiqueta que de verdad te importa.

> ⚠️ **Errores típicos**
> - **Olvidar el `ELSE`.** Si ninguna condición se cumple y no hay `ELSE`, esa fila
>   queda en `NULL` (vacío) sin que te des cuenta.
> - **Orden de las condiciones al revés.** Si pones primero `< 300` y luego `< 70`,
>   *nada* entrará nunca en la de 70: la primera ya se los llevó a todos.
> - **Olvidar `END` o el `AS`.** Todo `CASE` se cierra con `END`, y conviene nombrar el
>   resultado con `AS`.


### ✍️ Ejercicio 2 — Clasificar cada sismo por profundidad
Devuelve `id`, `lugar`, `profundidad_km` y una columna nueva `profundidad_tipo` que valga:
`'superficial'` si la profundidad es menor a 70 km, `'intermedio'` si está entre 70 y
300, y `'profundo'` si es mayor a 300. Ordena por `id`. Completa `consulta_2`.

*(Pista geológica: en nuestra muestra real verás solo superficiales e intermedios; los
sismos muy profundos son menos frecuentes.)*


In [ ]:
consulta_2 = """
-- TODO: reemplaza el SELECT de abajo.
-- Devuelve id, lugar, profundidad_km y una columna nueva profundidad_tipo
-- construida con CASE WHEN:
--   < 70 km          -> 'superficial'
--   entre 70 y 300   -> 'intermedio'
--   > 300 km         -> 'profundo'
-- Recuerda cerrar con END AS profundidad_tipo. Ordena por id.
SELECT * FROM sismos
"""
correr(consulta_2)

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ───────────────────────────────────────────
try:
    df = correr(consulta_2)
    assert "profundidad_tipo" in df.columns, "Falta la columna profundidad_tipo (cierra con END AS ...)."
    assert len(df) == 15, f"Deben salir 15 filas (una por sismo), no {len(df)}."
    conteo = df["profundidad_tipo"].value_counts().to_dict()
    assert conteo.get("superficial", 0) == 6, \
        f"Deberian ser 6 'superficial' (<70 km), salieron {conteo.get('superficial', 0)}."
    assert conteo.get("intermedio", 0) == 9, \
        f"Deberian ser 9 'intermedio' (70-300 km), salieron {conteo.get('intermedio', 0)}."
    print("✅ Correcto: clasificaste cada sismo con una feature condicional (CASE WHEN).")
except Exception as e:
    print("❌ Aun no. Revisa tu consulta_2 y vuelve a intentar.")
    print("   Pista:", e)

## 4. *Features* temporales con funciones de ventana (`LAG`)

La Línea B se especializa en **series temporales**, así que esta sección es el corazón
del módulo. Muchas veces una *feature* poderosa es **"¿qué pasó antes?"**: la magnitud
del sismo anterior, las ventas del mes pasado, la temperatura de ayer. A estas variables
se les llama de **rezago** (*lag*).

Para mirar "la fila anterior" sin perder la fila actual se usan las **funciones de
ventana**. La estrella aquí es `LAG`:

```sql
LAG(magnitud) OVER (ORDER BY fecha_hora) AS magnitud_anterior
```

Léelo así: *"ordena las filas por `fecha_hora` y, en cada fila, tráeme la `magnitud` de
la fila inmediatamente anterior"*. La **primera** fila no tiene anterior, así que su
valor queda en `NULL`. A diferencia de `GROUP BY`, la ventana **no colapsa** las filas:
mantiene una fila por sismo y le agrega la columna mirando a sus vecinas.

> 🧠 **Analogía.** Es como ir en una fila del banco anotando, para cada persona, cuánto
> demoró **la persona de adelante**. No achicas la fila; solo agregas una nota mirando
> hacia atrás.

### 🚨 *Data leakage* (fuga de información): la trampa #1 de la ciencia de datos
Cuando fabricas *features* para predecir el futuro, **solo puedes usar información que
ya existía en ese momento**. Mirar "hacia atrás" con `LAG` es legítimo: el sismo anterior
ya ocurrió. Pero si usaras `LEAD` (mirar la fila *siguiente*) para predecir el sismo
actual, le estarías "soplando" al modelo el futuro: eso es *data leakage*. En
entrenamiento parece magia (acierta todo); en la realidad fracasa, porque el futuro aún
no ha pasado. **Regla de oro: una *feature* nunca debe contener información del momento
que intentas predecir o posterior a él.**

> ⚠️ **Errores típicos**
> - **Olvidar el `ORDER BY` dentro del `OVER`.** Sin un orden explícito, "anterior" no
>   significa nada y el resultado es impredecible.
> - **Confundir `LAG` (anterior) con `LEAD` (siguiente).** `LEAD` mira al futuro: úsalo
>   con cuidado, casi nunca como *feature* predictiva.
> - **Asustarse por el `NULL` de la primera fila.** Es normal y correcto: la primera
>   observación no tiene pasado.


### ✍️ Ejercicio 3 — La magnitud del sismo anterior
Ordena los sismos del más antiguo al más reciente (`ORDER BY fecha_hora`) y agrega la
columna `magnitud_anterior` con la magnitud del sismo inmediatamente anterior, usando
`LAG`. Devuelve `id`, `fecha_hora`, `magnitud` y `magnitud_anterior`. Completa `consulta_3`.


In [ ]:
consulta_3 = """
-- TODO: reemplaza el SELECT de abajo.
-- Ordena los sismos del mas antiguo al mas reciente (ORDER BY fecha_hora)
-- y agrega una columna magnitud_anterior con la magnitud del sismo
-- INMEDIATAMENTE anterior, usando la funcion de ventana LAG:
--   LAG(magnitud) OVER (ORDER BY fecha_hora) AS magnitud_anterior
-- Devuelve: id, fecha_hora, magnitud, magnitud_anterior.
SELECT * FROM sismos
"""
correr(consulta_3)

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ───────────────────────────────────────────
try:
    df = correr(consulta_3)
    assert "magnitud_anterior" in df.columns, "Falta la columna magnitud_anterior (usa LAG)."
    assert len(df) == 15, f"Deben salir 15 filas, no {len(df)}."
    df = df.sort_values("fecha_hora").reset_index(drop=True)
    assert pd.isna(df.loc[0, "magnitud_anterior"]), \
        "El primer sismo (el mas antiguo) no tiene anterior: magnitud_anterior debe ser NULL/NaN."
    assert abs(float(df.loc[1, "magnitud_anterior"]) - 2.9) < 1e-6, \
        "El segundo sismo debe heredar 2.9 (la magnitud del primero). Revisa el ORDER BY del LAG."
    print("✅ Correcto: creaste una feature temporal de rezago (lag) con una funcion de ventana.")
except Exception as e:
    print("❌ Aun no. Revisa tu consulta_3 y vuelve a intentar.")
    print("   Pista:", e)

## 5. Juntar todo: la **tabla analítica**

Ya sabes agregar, clasificar y mirar al pasado. El paso final de un *data scientist* es
**reunir varias *features* en una sola tabla con una fila por entidad**. Esa es la
**tabla analítica (ABT)**: el producto que de verdad se le entrega a un modelo.

Un truco muy usado combina las dos primeras ideas: contar cuántas filas cumplen una
condición, dentro de cada grupo, con **`SUM(CASE WHEN … THEN 1 ELSE 0 END)`**. Cada fila
que cumple aporta un `1`; el `SUM` los suma. Así obtienes, por región, "cuántos sismos
superficiales" o "cuántos profundos" en columnas separadas.

```sql
SUM(CASE WHEN profundidad_km < 70 THEN 1 ELSE 0 END) AS n_superficiales
```

> 🧠 **Analogía.** Es levantar la mano: por cada sismo superficial alguien levanta la
> mano (un `1`); al final cuentas las manos levantadas de cada región.


### ✍️ Ejercicio 4 — Tu primera tabla analítica
Construye una tabla con **una fila por región** y estas columnas:
`region`, `n_sismos`, `magnitud_promedio`, `n_superficiales`, `n_profundos`, donde
`n_superficiales` cuenta los sismos con profundidad menor a 70 km y `n_profundos` los de
70 km o más. Ordena por `n_sismos` de mayor a menor. Completa `consulta_4`.

Cuando salga ✅, mira la tabla: **eso** es lo que en el próximo módulo (D9 ·
Fundamentos de Machine Learning) le entregarás a tu primer modelo.


In [ ]:
consulta_4 = """
-- TODO: reemplaza el SELECT de abajo.
-- Arma la TABLA ANALITICA: una fila por region con estas columnas (nombres exactos):
--   region, n_sismos, magnitud_promedio, n_superficiales, n_profundos
-- n_superficiales = cuantos sismos de esa region tienen profundidad < 70 km
-- n_profundos     = cuantos tienen profundidad >= 70 km
-- Truco: SUM(CASE WHEN condicion THEN 1 ELSE 0 END) cuenta los que cumplen.
-- Usa GROUP BY region y ordena por n_sismos DESC.
SELECT * FROM sismos
"""
correr(consulta_4)

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ───────────────────────────────────────────
try:
    df = correr(consulta_4)
    esperadas = {"region", "n_sismos", "magnitud_promedio", "n_superficiales", "n_profundos"}
    assert esperadas.issubset(set(df.columns)), \
        f"Faltan columnas. Necesito exactamente: {sorted(esperadas)}"
    assert len(df) == 6, f"Una fila por region: deben ser 6, no {len(df)}."
    assert int(df["n_sismos"].sum()) == 15, "n_sismos debe sumar 15."
    assert int(df["n_superficiales"].sum()) == 6, "En total hay 6 sismos superficiales (<70 km)."
    assert int(df["n_profundos"].sum()) == 9, "En total hay 9 sismos de 70 km o mas."
    print("✅ Correcto: armaste la TABLA ANALITICA, lista para alimentar un modelo. 🎯")
except Exception as e:
    print("❌ Aun no. Revisa tu consulta_4 y vuelve a intentar.")
    print("   Pista:", e)

## 6. Cierre

Hoy dejaste de *consultar* datos para empezar a *fabricar variables*. Aprendiste a:

1. **Agregar** muchas filas en *features* por entidad con `GROUP BY` (`COUNT`, `AVG`, `MAX`).
2. **Clasificar** con `CASE WHEN` para crear etiquetas y banderas.
3. Crear *features* **temporales de rezago** con la función de ventana `LAG`, y a
   cuidarte del **data leakage**.
4. **Reunir** todo en una **tabla analítica**: una fila por entidad, lista para un modelo.

Esa tabla analítica es el puente hacia la segunda mitad de la Línea B: en **D9** la usarás
para entrenar tu primer modelo de Machine Learning.

### Mini-glosario
- ***Feature*:** columna que describe una entidad y que un modelo puede usar.
- **Entidad:** la unidad de análisis; tendrá una fila en la tabla final (aquí, la región).
- **Tabla analítica (ABT):** una fila por entidad, una columna por *feature*.
- ***Lag* (rezago):** valor de una observación anterior usado como *feature*.
- ***Data leakage*:** usar información del futuro (o del propio resultado) como *feature*;
  infla el rendimiento en pruebas y arruina el modelo en la realidad.

---
*Fuente de datos: Centro Sismológico Nacional (CSN), Universidad de Chile —
[sismologia.cl](https://www.sismologia.cl/). Snapshot del 2026-06-18.*
*Contenido bajo licencia CC BY 4.0 · Formación Pública.*
